In [598]:
import torch 
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

In [599]:
class LstmClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, bidirectional = True):
        super().__init__()

        self.input_sz = input_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,bidirectional=bidirectional, dropout=0.2)
        self.clf = nn.Sequential(
            nn.Linear(2 * hidden_size, 4 * hidden_size),
            nn.LayerNorm(4 * hidden_size),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(4 * hidden_size, 7)
        )

    def forward(self, x, lengths):
        B, T, F = x.shape
        packed_x = pack_padded_sequence(x, lengths=lengths.cpu(), batch_first=True, enforce_sorted=False)

        output, (h, c) = self.lstm(packed_x) # output -> T, B, 2*H, h -> B, 2*H, c -> B, 2*H
        clf_tensor = torch.cat([h[-2], h[-1]], dim=-1)
        logits = self.clf(clf_tensor)

        return logits



In [600]:
import pandas as pd

In [601]:
df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

In [602]:
df = df_test.groupby('IDSample')

In [603]:
subtask1 = []

In [604]:
for id, gr in df:
    subtask1.append(gr['FrameNumber'].max())

In [605]:
df_train.head()

,IDSample,FrameNumber,J1X,J1Y,J1Z,J2X,J2Y,J2Z,J3X,J3Y,...,J23Y,J23Z,J24X,J24Y,J24Z,J25X,J25Y,J25Z,Action,Camera
0,1,1,0.164490,0.291355,3.091319,0.189296,0.615010,3.080443,0.209298,0.928650,...,0.249604,2.929706,0.237679,0.185178,2.881282,0.290780,0.236038,2.852048,3,1
1,1,2,0.164051,0.291135,3.091265,0.187257,0.614560,3.080929,0.206220,0.927991,...,0.242949,2.934857,0.237713,0.184868,2.881657,0.288819,0.236270,2.851250,3,1
2,1,3,0.160394,0.291922,3.091712,0.183053,0.614688,3.081785,0.202935,0.927368,...,0.261962,2.940350,0.237603,0.182165,2.880324,0.286495,0.221468,2.868750,3,1
3,1,4,0.157784,0.291979,3.090791,0.179416,0.614119,3.081133,0.198786,0.925982,...,0.257649,2.934111,0.231934,0.184934,2.879659,0.289556,0.229324,2.863987,3,1
4,1,5,0.154023,0.289773,3.091381,0.174946,0.611712,3.081385,0.194107,0.923457,...,0.252397,2.932880,0.223781,0.178191,2.879539,0.278711,0.216850,2.864786,3,1


In [606]:
y_train = df_train[['IDSample','Action', 'Camera']]

In [607]:
X_train = df_train.drop(columns = ['Action', 'Camera'])
X_test = df_test

In [608]:
X_test.head()

,IDSample,FrameNumber,J1X,J1Y,J1Z,J2X,J2Y,J2Z,J3X,J3Y,...,J22Z,J23X,J23Y,J23Z,J24X,J24Y,J24Z,J25X,J25Y,J25Z
0,3361,1,-0.707474,-0.558262,3.659323,-0.712082,-0.224231,3.667649,-0.713740,0.105216,...,3.545863,-0.794203,-0.362780,3.570667,-0.579403,-0.309495,3.535433,-0.504720,-0.324692,3.571167
1,3361,2,-0.701481,-0.555497,3.653012,-0.708182,-0.223555,3.663723,-0.711626,0.104554,...,3.548151,-0.811219,-0.366514,3.571125,-0.578761,-0.305720,3.534078,-0.510337,-0.321618,3.561222
2,3361,3,-0.699060,-0.555092,3.648376,-0.703081,-0.221451,3.657866,-0.703695,0.108269,...,3.541340,-0.804400,-0.374341,3.560300,-0.580567,-0.325120,3.524870,-0.513830,-0.323958,3.553727
3,3361,4,-0.692364,-0.554330,3.643446,-0.697653,-0.221149,3.653842,-0.699972,0.107870,...,3.529219,-0.813160,-0.380458,3.553818,-0.577347,-0.340454,3.514266,-0.514556,-0.339562,3.545001
4,3361,5,-0.690229,-0.554802,3.639278,-0.693909,-0.220811,3.649389,-0.694498,0.108867,...,3.524888,-0.801957,-0.374812,3.546375,-0.574917,-0.336967,3.497991,-0.505000,-0.345860,3.545715


In [609]:
ids = X_test.IDSample.unique()

In [610]:
import numpy as np

In [611]:
sequences_train = df_train.groupby(by='IDSample')
sequences_test = X_test.groupby(by='IDSample')

In [612]:
for id, gr in sequences_train:
    print(np.array(gr.drop(columns=['FrameNumber', 'IDSample'])).mean(axis = 0))
    break

[ 0.11652088  0.32643804  3.03202005  0.13438514  0.64838321  3.02755837
  0.15099264  0.95976793  3.01048254  0.11354451  1.09134258  3.03140586
  0.02486561  0.84653266  3.11254686  0.00965282  0.58110601  3.02568225
 -0.03324416  0.3457645   2.97468496 -0.01988331  0.31032946  2.96313118
  0.22827787  0.79913363  2.88459919  0.29816289  0.53658051  2.8254846
  0.25397983  0.32283666  2.79316716  0.22635254  0.2849953   2.79462632
  0.06827117  0.30486552  3.0611524   0.13222708 -0.06681493  3.11160905
  0.20009956 -0.43023999  3.21457826  0.15940487 -0.50072864  3.12614089
  0.16167286  0.33943529  2.9226943  -0.02838612  0.05830812  2.80806886
  0.10096582 -0.25732273  2.97147802  0.04651932 -0.33409661  2.90295511
  0.14703849  0.88345183  3.01708289 -0.02923771  0.26458732  2.93917902
 -0.01200594  0.31517549  2.93113482  0.20234824  0.22494186  2.80016549
  0.26296331  0.27259341  2.78972214  3.          1.        ]


In [613]:
from torch.utils.data import Dataset, DataLoader, random_split

In [614]:
class lstmdata(Dataset):
    def __init__(self, sequences, label = False):
        super().__init__()

        self.seq = []
        self.labels = [] if label else None 

        if label is not False:
            for id, gr in sequences:
                    array = np.array(gr.drop(columns=['FrameNumber', 'IDSample', 'Action', 'Camera']))
                    T = array.shape[0]

                    mean = np.repeat(array.mean(axis=0, keepdims=True), T, axis=0)
                    max_ = np.repeat(array.max(axis=0, keepdims=True), T, axis=0)
                    min_ = np.repeat(array.min(axis=0, keepdims=True), T, axis=0)
                    std = np.repeat(array.std(axis=0, keepdims=True), T, axis=0)

                    array = np.concatenate(
                        [array, mean, max_, min_, std],
                        axis=1
                    )

                    self.seq.append(
                        torch.tensor(array, dtype=torch.float32)
                    )
                    grup = gr.drop(columns=['FrameNumber', 'IDSample'])
                    self.labels.append([grup['Action'].iloc[0], grup['Camera'].iloc[0]])
        else:
            for id, gr in sequences:
                array = np.array(gr.drop(columns=['FrameNumber', 'IDSample']))

                T = array.shape[0]

                mean = np.repeat(array.mean(axis=0, keepdims=True), T, axis=0)
                max_ = np.repeat(array.max(axis=0, keepdims=True), T, axis=0)
                min_ = np.repeat(array.min(axis=0, keepdims=True), T, axis=0)
                std = np.repeat(array.std(axis=0, keepdims=True), T, axis=0)

                array = np.concatenate(
                    [array, mean, max_, min_, std],
                    axis=1
                )

                self.seq.append(
                    torch.tensor(array, dtype=torch.float32)
                )
                
    def __len__(self):
        return len(self.seq)

    def __getitem__(self, index):

        seq = self.seq[index]

        if self.labels is not None:
            return seq, self.labels[index][0], self.labels[index][1]
        return seq

In [615]:
len(y_train)

313762

In [616]:
train_data = lstmdata(sequences_train, True)
test_data = lstmdata(sequences_test)

In [617]:
len(train_data[0][0])

57

In [618]:
train_data, valid_data = random_split(train_data,[int(len(train_data)*0.8), len(train_data) - int(len(train_data)*0.8)])

In [619]:
def collate_fn(batch):
    seq, actions, cameras = zip(*batch)
    
    lengths = torch.tensor([len(s) for s in seq])

    padded_seq = pad_sequence(
        seq,
        batch_first=True,
        padding_value=0
    )

    return padded_seq, lengths, torch.tensor(actions), torch.tensor(cameras)

In [620]:
def collate_fn1(batch):
    seq = batch
    
    lengths = torch.tensor([len(s) for s in seq])

    padded_seq = pad_sequence(
        seq,
        batch_first=True,
        padding_value=0
    )

    return padded_seq, lengths

In [621]:
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn = collate_fn, drop_last=True)
valid_dataloader = DataLoader(valid_data, batch_size=64, collate_fn = collate_fn, shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=64, collate_fn = collate_fn1, shuffle=False)

In [622]:
next(iter(train_dataloader))

(tensor([[[ 0.0466,  0.0911,  3.4064,  ...,  0.1779,  0.1738,  0.1415],
          [ 0.0459,  0.0913,  3.4068,  ...,  0.1779,  0.1738,  0.1415],
          [ 0.0457,  0.0903,  3.4072,  ...,  0.1779,  0.1738,  0.1415],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
 
         [[-0.4671, -0.3683,  2.5649,  ...,  0.2049,  0.2785,  0.0610],
          [-0.4709, -0.3688,  2.5738,  ...,  0.2049,  0.2785,  0.0610],
          [-0.4712, -0.3686,  2.5735,  ...,  0.2049,  0.2785,  0.0610],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
 
         [[ 0.2487, -0.0155,  3.6434,  ...,  0.0265,  0.0684,  0.0293],
          [ 0.2500, -0.0127,

In [623]:
epochs = 35
model = LstmClassifier(375, 128, 2, True)
optimizer = torch.optim.AdamW(model.parameters(),lr = 1e-3, weight_decay = 1e-4)
loss_action = torch.nn.CrossEntropyLoss()
loss_camera = torch.nn.CrossEntropyLoss()

In [624]:
from tqdm import tqdm
device = "mps" if torch.mps.is_available() else "cpu"
model = model.to(device)

In [625]:
torch.arange(5)[4]

tensor(4)

In [626]:
from sklearn.metrics import accuracy_score as accuracy

In [627]:
for epoch in range(epochs):
    train_losses = []
    valid_losses = []


    model.train()
    train_bar = tqdm(train_dataloader, desc = "Training..")
    total_train_loss = 0.
    train_preds = []
    valid_preds = []

    train_true = []
    valid_true = []

    for X, lengths, actions, cameras in train_bar:

        optimizer.zero_grad()
        X, lengths, actions, cameras = X.to(device), lengths.to(device), actions.to(device), cameras.to(device)

        y_hat = model(X,lengths)
        loss_a = loss_action(y_hat[:, :4], actions)
        loss_c = loss_camera(y_hat[:, 4:], cameras)

        train_preds.append(torch.argmax(y_hat[:,:4], dim = 1))
        train_true.append(actions)

        loss = loss_a + loss_c
        total_train_loss+=loss.item()
        train_losses.append(loss.item())

        train_bar.set_postfix(loss = loss.item())
        loss.backward()
        optimizer.step()
    avg_train_loss = total_train_loss / len(train_dataloader)
    train_preds = torch.cat(train_preds)
    train_true = torch.cat(train_true)
    print(f"training epoch {epoch} has avg loss {avg_train_loss}")
    print(f"training accuracy on epoch {epoch} is {accuracy(train_true.detach().cpu().numpy(), train_preds.detach().cpu().numpy())}")


    model.eval()
    valid_bar = tqdm(valid_dataloader, desc = "Validating..")
    total_valid_loss = 0.
    with torch.inference_mode():
        for X, lengths, actions, cameras in valid_bar:
        
                X, lengths, actions, cameras = X.to(device), lengths.to(device), actions.to(device), cameras.to(device)
        
                y_hat = model(X,lengths)
                loss_a = loss_action(y_hat[:, :4], actions)
                loss_c = loss_camera(y_hat[:, 4:], cameras)

                valid_preds.append(torch.argmax(y_hat[:,:4], dim = 1))
                valid_true.append(actions)
        
                loss = loss_a + loss_c
                total_valid_loss+=loss.item()
                valid_losses.append(loss.item())
        
                valid_bar.set_postfix(loss = loss.item())
    avg_valid_loss = total_valid_loss / len(valid_dataloader)
    valid_preds = torch.cat(valid_preds)
    valid_true = torch.cat(valid_true)
    print(f"validating epoch {epoch} has avg loss {avg_valid_loss}")
    print(f"validation accuracy on epoch {epoch} is {accuracy(valid_true.detach().cpu().numpy(), valid_preds.detach().cpu().numpy())}")

    

Training..: 100%|██████████| 41/41 [00:16<00:00,  2.55it/s, loss=1.27]


training epoch 0 has avg loss 1.5555332201283152
training accuracy on epoch 0 is 0.2629573170731707


Validating..: 100%|██████████| 11/11 [00:01<00:00, 10.24it/s, loss=1.31]


validating epoch 0 has avg loss 1.194651560349898
validation accuracy on epoch 0 is 0.4008941877794337


Training..: 100%|██████████| 41/41 [00:14<00:00,  2.79it/s, loss=1.05] 


...

training epoch 23 has avg loss 0.49606669457947333
training accuracy on epoch 23 is 0.7614329268292683


Validating..: 100%|██████████| 11/11 [00:01<00:00,  9.28it/s, loss=0.639]


validating epoch 23 has avg loss 0.5978595560247247
validation accuracy on epoch 23 is 0.7004470938897168


Training..:  32%|███▏      | 13/41 [00:05<00:12,  2.28it/s, loss=0.577]


KeyboardInterrupt: 

In [ ]:
next(iter(test_dataloader))

(tensor([[[-0.7075, -0.5583,  3.6593,  ...,  3.7765, -1.0362,  1.9723],
          [-0.7015, -0.5555,  3.6530,  ...,  3.7742, -0.9980,  1.9688],
          [-0.6991, -0.5551,  3.6484,  ...,  3.7629, -1.0012,  1.9617],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
 
         [[-0.1298, -0.4527,  2.7718,  ...,  2.8853, -1.0341,  1.4052],
          [-0.1298, -0.4531,  2.7720,  ...,  2.8856, -1.0480,  1.4024],
          [-0.1300, -0.4531,  2.7720,  ...,  2.8858, -1.0352,  1.4052],
          ...,
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]],
 
         [[ 0.1541,  0.2641,  3.5932,  ...,  3.7937, -0.5460,  1.6064],
          [ 0.1376,  0.2605,

In [628]:
model.eval()
test_bar = tqdm(test_dataloader, desc = "Testing..")
total_valid_loss = 0.
Actions = []
Cameras = []
with torch.inference_mode():
    for X, lengths in test_bar:
        
            X, lengths = X.to(device), lengths.to(device)
        
            y_hat = model(X,lengths)

            Actions.append(torch.argmax(y_hat[:,:4], dim = 1))
            Cameras.append(torch.argmax(y_hat[:,4:], dim = 1))

Testing..: 100%|██████████| 22/22 [00:02<00:00,  9.61it/s]


In [629]:
Actions = torch.cat(Actions)

In [630]:
Cameras = torch.cat(Cameras)

In [631]:
print(Cameras)

tensor([1, 1, 1,  ..., 1, 1, 1], device='mps:0')


In [632]:
subtask1

[np.int64(124),
 np.int64(78),
 np.int64(207),
 np.int64(66),
 np.int64(83),
 np.int64(65),
 np.int64(86),
 np.int64(100),
 np.int64(87),
 np.int64(90),
 np.int64(161),
 np.int64(86),
 np.int64(71),
 np.int64(200),
 np.int64(78),
 np.int64(139),
 np.int64(58),
 np.int64(51),
 np.int64(91),
 np.int64(139),
 np.int64(75),
 np.int64(71),
 np.int64(88),
 np.int64(66),
 np.int64(115),
 np.int64(79),
 np.int64(66),
 np.int64(68),
 np.int64(81),
 np.int64(67),
 np.int64(181),
 np.int64(71),
 np.int64(95),
 np.int64(67),
 np.int64(158),
 np.int64(71),
 np.int64(132),
 np.int64(81),
 np.int64(110),
 np.int64(65),
 np.int64(79),
 np.int64(138),
 np.int64(87),
 np.int64(74),
 np.int64(107),
 np.int64(59),
 np.int64(46),
 np.int64(78),
 np.int64(50),
 np.int64(198),
 np.int64(80),
 np.int64(78),
 np.int64(102),
 np.int64(86),
 np.int64(74),
 np.int64(81),
 np.int64(68),
 np.int64(72),
 np.int64(67),
 np.int64(72),
 np.int64(64),
 np.int64(60),
 np.int64(125),
 np.int64(55),
 np.int64(83),
 np.int6

In [633]:
submission = pd.read_csv("sample_output.csv")

In [634]:
submission

,subtaskID,datapointID,answer
0,1,3361,0
1,2,3361,0
2,3,3361,0
3,1,3362,0
4,2,3362,0
...,...,...,...
4129,2,4737,0
4130,3,4737,0
4131,1,4738,0
4132,2,4738,0


In [635]:
Actions[0].item()

2

In [636]:
answer = []
for i in range(len(Actions)):
    answer.append(subtask1[i])
    answer.append(Actions[i].item())
    answer.append(Cameras[i].item())

In [637]:
submission['answer'] = answer

In [638]:
submission.to_csv("Submission.csv", index = False)